## Data overview

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("../data/processed/final_enriched_rental_data.csv")
data.head()

,hour,location_id,direct_count,registered_count,total_count,is_zero_filled_rental,date,hour_of_day,day_of_week,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_missing_weather,holiday,is_holiday,is_workday
0,2011-01-01 00:00:00,0,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,Not a holiday,0,0
1,2011-01-01 00:00:00,1,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,Not a holiday,0,0
2,2011-01-01 00:00:00,2,1,0,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,Not a holiday,0,0
3,2011-01-01 00:00:00,3,0,1,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,Not a holiday,0,0
4,2011-01-01 00:00:00,4,0,2,2,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,Not a holiday,0,0


In [ ]:
### add helper features for EDA
data["hour"] = pd.to_datetime(data["hour"])
data["quarter"] = data["hour"].dt.quarter # season




In [ ]:
data.columns

In [10]:
data.shape
# data.info()
data.describe()
# data.isna().sum()

,location_id,direct_count,registered_count,total_count,is_zero_filled_rental,hour_of_day,day_of_week,month,is_weekend,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_missing_weather,is_holiday,is_workday
count,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000,368424.000000
mean,10.000000,1.682890,7.254310,8.937200,0.152102,11.500000,3.002736,6.519836,0.287278,15.262896,15.281122,62.842767,12.794374,0.009405,0.028728,0.683995
std,6.055309,2.662112,7.674525,9.121555,0.359120,6.922196,2.003418,3.449556,0.452493,9.079545,11.385721,19.303595,8.234173,0.096522,0.167041,0.464916
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,-7.100000,-16.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,0.000000,1.000000,2.000000,0.000000,5.750000,1.000000,4.000000,0.000000,8.000000,6.000000,48.000000,7.000000,0.000000,0.000000,0.000000
50%,10.000000,1.000000,5.000000,6.000000,0.000000,11.500000,3.000000,7.000000,0.000000,15.500000,16.000000,63.000000,13.000000,0.000000,0.000000,1.000000
75%,15.000000,2.000000,11.000000,13.000000,0.000000,17.250000,5.000000,10.000000,1.000000,23.000000,25.000000,79.000000,17.000000,0.000000,0.000000,1.000000
max,20.000000,28.000000,59.000000,64.000000,1.000000,23.000000,6.000000,12.000000,1.000000,39.000000,50.000000,100.000000,57.000000,1.000000,1.000000,1.000000


## Missing flags check
### is_missing_rental

In [11]:
## 检查 0 是否主要来自 missing rental
data["is_zero_filled_rental"].value_counts()

is_zero_filled_rental
0    312386
1     56038
Name: count, dtype: int64

In [12]:
pd.crosstab(
    data["total_count"] == 0,
    data["is_zero_filled_rental"]
)

is_zero_filled_rental,0,1
total_count,,
False,312386,0
True,0,56038


In [ ]:
data.groupby("is_missing_rental")["rental_count"].agg(
    ["count", "mean", "median", "min", "max"]
)

In [ ]:
data.groupby("is_missing_weather")["rental_count"].agg(
    ["count", "mean", "median", "min", "max"]
)

In [ ]:
zero_count = (data["rental_count"] == 0).sum()
total_count = len(data)

zero_ratio = zero_count / total_count

zero_count, total_count, zero_ratio

有多少行 rental_count 是 0？
0 占整个数据集的比例是多少？

In [ ]:
data.groupby("is_missing_rental")["rental_count"].describe()
判断：真实观测 rental rows 和补全 rental rows 的 target 分布是否完全不同？

Rows with `is_missing_rental = 1` were added when creating a complete hourly-location grid. Their rental counts were filled with 0, which means that these rows represent hours where no rental event was recorded.

## is_missing-weather
EDA 中使用 is_missing_weather 检查数据质量
模型训练中暂时不使用它




In [ ]:
data["is_missing_weather"].value_counts()
data.groupby("is_missing_weather")["rental_count"].agg(["count", "mean"])
data.groupby("is_missing_weather")[["temperature", "humidity", "wind_speed"]].mean()

## Tartge distribution

For most EDA and modeling-related checks, I use the full dataset because it represents the final model input. However, I also create an `observed_rentals` subset to compare patterns only for rows where rental activity was originally observed.

In [ ]:
full_data = data.copy()
observed_rentals = data[data["is_missing_rental"] == 0].copy()
#For most EDA and modeling-related checks, I use the full dataset because it represents the final model input. However, I also create an `observed_rentals` subset to compare patterns only for rows where rental activity was originally observed.

In [6]:
data["total_count"].value_counts()

total_count
0     56038
1     33751
2     24199
3     20437
4     18663
      ...  
58        6
62        5
61        3
59        2
64        1
Name: count, Length: 64, dtype: int64

In [ ]:
## Reantal type composition: pie chart 或 bar chart

## time-based:  3. weekday-hour heatmap

早晚高峰是否更高？
周末和工作日需求是否不同？
不同月份是否有季节变化？

1. hourly demand (total to show peak hour, registered/pickup to show time pattern),
早晚高峰是否更高？
registered 和 pickup 是否有不同小时模式？
line chart 或 bar chart
data.groupby("hour")[["registered_count", "pickup_count", "rental_count"]].mean()

In [ ]:
2. weekday/weekend demand,
平均总需求： 周末平均每小时需求是否更高？
不同 rental type 的平均需求：registered 和 pickup 在周末/工作日是否有不同模式？

In [ ]:
3. weekday × hour 的 average rental demand heatmap
一周中哪一天、哪个小时租车需求最高？
是否存在明显通勤高峰？
周末的需求峰值是否从早晚变成中午/下午？

## weather effect: temperature, wind speed, humidity, condition

温度越高租车越多吗？
下雨时租车需求是否下降？
风大时是否需求下降？
不同天气 condition 下需求是否不同？

### conditian 单独分析：
1. 看原始类别：
data["conditions"].value_counts()


3. 看每种天气下的平均租车量：

data.groupby("condition")["rental_count"].agg(
    ["count", "mean", "median", "sum"]
).sort_values("mean", ascending=False)

建议重点看：
count  → 这个天气类别出现了多少次
mean   → 这种天气下平均每小时租车需求
median → 是否被极端值影响
sum    → 全年总需求

4. encoding：(before model training)
data_encoded = pd.get_dummies(
    data,
    columns=["conditions"],
    drop_first=True
)

1. single weather feature
condition → rental_count
temperature → rental_count
humidity → rental_count
wind_speed → rental_count
precipitation → rental_count


2. group features
weather_category + hour
weather_category + is_weekend
temperature_bin + condition
condition + wind_bin

The original weather condition column is a categorical text feature. It cannot be used directly by most machine learning models, but it may contain useful information about overall weather conditions, such as clear, cloudy, rainy, snowy, or foggy weather.

For EDA, I simplify the original condition labels into broader weather categories. This makes the weather patterns easier to interpret and avoids having too many sparse categories.

For modeling, this categorical feature will need to be encoded, for example using one-hot encoding. It can be used together with numerical weather features such as temperature, humidity, wind speed, and precipitation.

In [ ]:
## location based

哪些 location 的全年需求最高？
不同 location 是否有不同使用模式？
某些 location 是否更偏向 registered rentals 或 pickup rentals？
bar chart
哪些 location 是高需求 location？
需求是否集中在少数 location？
如果 location 很多，可以只画 top 10：

top_locations = (
    data.groupby("location_id")["rental_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
## holiday based

节假日租车需求是否和平时不同？
节假日 registered 和 pickup 的比例是否变化？

In [ ]:
## heatmap


weekday × hour 的 average rental demand heatmap
location_id × hour
temperature_bin × hour
weather_condition × hour

In [ ]:
## condition encoding: 
numeric features & categorical features

In [ ]:
FEATURES = ["hour", "location_id", "date", "hour_of_day", "day_of_week", "mont", "is_weekend", "conditions",]